# Star Schema Data Model for Data

This notebook creates a dimensional data model (star schema) from the silver layer tables using **Apache Iceberg** table format.

## Architecture:

### Dimension Tables:
1. **dim_customer** - Customer dimension with demographics and registration info
2. **dim_product** - Product dimension with category, supplier, brand, and pricing info
3. **dim_date** - Date dimension for time-based analysis
4. **dim_warehouse** - Warehouse/location dimension for inventory analysis

### Fact Tables:
1. **fact_orders** - Order header information with totals and payment details
2. **fact_order_transactions** - Transactional grain combining order and line item details with enriched calculations
3. **fact_sales** - Sales transactions (from order_items_clean + orders_clean)
4. **fact_inventory** - Inventory snapshots by product and warehouse
5. **fact_reviews** - Product reviews and ratings
6. **fact_web_events** - Web clickstream and user behavior events

## Benefits:
* Optimized for analytical queries
* Simplified joins through surrogate keys
* Historical tracking with SCD Type 2 where applicable
* Consistent dimensional attributes across all facts
* **Apache Iceberg format** for ACID transactions, time travel, and schema evolution
* **No indexes required** - Iceberg manages optimization internally

In [0]:
-- Create the gold_dimensional schema
CREATE SCHEMA IF NOT EXISTS skofy27.gold_dimensional
COMMENT 'Dimensional model (star schema) for analytics and reporting';

In [0]:
use catalog skofy27;
use schema gold_dimensional;

In [0]:
-- Create Customer Dimension
CREATE OR REPLACE TABLE  dim_customer (
  customer_key BIGINT NOT NULL,
  customer_id INT NOT NULL,
  first_name STRING,
  last_name STRING,
  full_name STRING,
  email STRING,
  email_domain STRING,
  phone STRING,
  phone_cleaned STRING,
  birth_date DATE,
  age INT,
  age_group STRING,
  gender STRING,
  address_line1 STRING,
  address_line2 STRING,
  city STRING,
  state STRING,
  country STRING,
  postal_code STRING,
  registration_date DATE,
  registration_year INT,
  registration_month INT,
  customer_tier STRING,
  source_system STRING,
  -- SCD Type 2 columns
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN,
  -- Audit columns
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA 
COMMENT 'Customer dimension with SCD Type 2 for tracking changes';

In [0]:
-- Create Product Dimension
CREATE OR REPLACE TABLE  dim_product (
  product_key BIGINT NOT NULL,
  product_id INT NOT NULL,
  product_name STRING,
  product_name_clean STRING,
  description STRING,
  sku STRING,
  -- Category attributes
  category_id INT,
  category_name STRING,
  category_path STRING,
  -- Supplier attributes
  supplier_id INT,
  supplier_name STRING,
  -- Product attributes
  brand STRING,
  brand_category STRING,
  color STRING,
  size STRING,
  weight DECIMAL(13,1),
  weight_category STRING,
  dimensions STRING,
  -- Pricing attributes
  price DECIMAL(14,2),
  cost DECIMAL(15,2),
  profit_margin DECIMAL(23,2),
  price_tier STRING,
  -- Status attributes
  is_active BOOLEAN,
  product_lifecycle_stage STRING,
  created_date DATE,
  days_since_launch INT,
  -- SCD Type 2 columns
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN,
  -- Audit columns
  updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Product dimension with category, supplier, and pricing attributes';

In [0]:
-- Create Date Dimension
CREATE OR REPLACE TABLE  dim_date (
  date_key INT NOT NULL,
  date_value DATE NOT NULL,
  year INT,
  quarter INT,
  month INT,
  month_name STRING,
  week INT,
  day_of_month INT,
  day_of_week INT,
  day_of_week_name STRING,
  day_of_year INT,
  is_weekend BOOLEAN,
  is_holiday BOOLEAN,
  fiscal_year INT,
  fiscal_quarter INT,
  fiscal_month INT,
  -- Relative date flags
  is_current_day BOOLEAN,
  is_current_week BOOLEAN,
  is_current_month BOOLEAN,
  is_current_quarter BOOLEAN,
  is_current_year BOOLEAN
)
USING DELTA
COMMENT 'Date dimension for time-based analysis';

In [0]:
-- Create Warehouse Dimension
CREATE OR REPLACE TABLE  dim_warehouse (
  warehouse_key BIGINT NOT NULL,
  warehouse_id INT NOT NULL,
  warehouse_region STRING,
  -- SCD Type 2 columns
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN,
  -- Audit columns
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Warehouse dimension for inventory location analysis';

In [0]:
-- Create Sales Fact Table
CREATE OR REPLACE TABLE  fact_sales (
  sales_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  order_date_key INT,
  customer_key BIGINT,
  product_key BIGINT,
  -- Degenerate dimensions (transaction identifiers)
  order_id INT,
  order_item_id INT,
  -- Order header attributes
  order_status STRING,
  order_status_category STRING,
  payment_method STRING,
  payment_method_category STRING,
  payment_status STRING,
  same_billing_shipping BOOLEAN,
  -- Measures - Quantity
  quantity INT,
  quantity_tier STRING,
  -- Measures - Amounts
  unit_price DECIMAL(14,2),
  unit_cost DECIMAL(15,2),
  discount_amount DECIMAL(15,2),
  total_amount DECIMAL(28,2),
  item_profit DECIMAL(30,2),
  item_margin DECIMAL(35,2),
  -- Order level amounts
  order_tax_amount DECIMAL(38,2),
  order_shipping_cost INT,
  order_discount_amount DECIMAL(11,1),
  order_total_amount DECIMAL(38,2),
  order_net_amount DECIMAL(38,2),
  -- Flags
  is_discounted BOOLEAN,
  has_order_discount BOOLEAN,
  -- Audit columns
  created_at TIMESTAMP,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Sales fact table with order and order item details';

In [0]:
-- Create Inventory Fact Table
CREATE OR REPLACE TABLE  fact_inventory (
  inventory_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  snapshot_date_key INT,
  product_key BIGINT,
  warehouse_key BIGINT,
  -- Degenerate dimension
  inventory_id INT,
  -- Measures - Quantities
  quantity_on_hand INT,
  quantity_reserved INT,
  quantity_available INT,
  reorder_level INT,
  -- Measures - Metrics
  days_of_supply DECIMAL(14,1),
  -- Attributes
  stock_status STRING,
  turnover_category STRING,
  -- Flags
  needs_reorder BOOLEAN,
  overstocked BOOLEAN,
  -- Audit columns
  last_updated DATE,
  created_at DATE,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Inventory snapshot fact table by product and warehouse';

In [0]:
-- Create Reviews Fact Table
CREATE OR REPLACE TABLE  fact_reviews (
  review_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  review_date_key INT,
  customer_key BIGINT,
  product_key BIGINT,
  -- Degenerate dimension
  review_id INT,
  -- Measures - Ratings
  rating INT,
  rating_category STRING,
  sentiment_score DECIMAL(2,1),
  sentiment_category STRING,
  helpful_votes INT,
  helpfulness_tier STRING,
  -- Text attributes
  review_text STRING,
  review_text_length INT,
  -- Time attributes
  days_since_purchase INT,
  review_recency STRING,
  review_year INT,
  review_month INT,
  -- Flags
  verified_purchase BOOLEAN,
  -- Audit columns
  created_at TIMESTAMP,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Product reviews and ratings fact table';

In [0]:
-- Create Web Events Fact Table
CREATE OR REPLACE TABLE  fact_web_events (
  event_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  event_date_key INT,
  customer_key BIGINT,
  product_key BIGINT,
  -- Degenerate dimensions
  event_id INT,
  session_id STRING,
  -- Event attributes
  event_type STRING,
  event_category STRING,
  event_hour INT,
  event_day_of_week STRING,
  -- Page attributes
  page_url STRING,
  page_category STRING,
  product_category STRING,
  -- Technical attributes
  ip_address STRING,
  user_agent STRING,
  device_type STRING,
  browser STRING,
  operating_system STRING,
  -- Referrer attributes
  referrer STRING,
  referrer_category STRING,
  -- Session attributes
  session_duration_bucket STRING,
  -- Flags
  is_mobile BOOLEAN,
  -- Audit columns
  event_timestamp TIMESTAMP,
  created_at TIMESTAMP,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Web clickstream events fact table';

In [0]:
-- Create Orders Fact Table
CREATE OR REPLACE TABLE  fact_orders (
  order_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  order_date_key INT,
  customer_key BIGINT,
  -- Degenerate dimension
  order_id INT,
  -- Order attributes
  order_status STRING,
  order_status_category STRING,
  payment_method STRING,
  payment_method_category STRING,
  payment_status STRING,
  order_year INT,
  order_month INT,
  order_quarter INT,
  order_day_of_week STRING,
  order_hour INT,
  -- Address attributes
  shipping_address STRING,
  billing_address STRING,
  same_billing_shipping BOOLEAN,
  -- Measures - Amounts
  total_amount DECIMAL(38,2),
  tax_amount DECIMAL(38,2),
  shipping_cost INT,
  discount_amount DECIMAL(11,1),
  net_amount DECIMAL(38,2),
  -- Derived measures
  order_value_tier STRING,
  discount_percentage DECIMAL(19,2),
  -- Flags
  has_discount BOOLEAN,
  -- Audit columns
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Orders fact table with order header information';

In [0]:
-- Create Order Transactions Fact Table (Transactional Grain)
CREATE OR REPLACE TABLE  fact_order_transactions (
  transaction_key BIGINT NOT NULL,
  -- Foreign keys to dimensions
  order_date_key INT,
  customer_key BIGINT,
  product_key BIGINT,
  -- Degenerate dimensions
  order_id INT,
  order_item_id INT,
  -- Order header attributes
  order_status STRING,
  order_status_category STRING,
  payment_method STRING,
  payment_method_category STRING,
  payment_status STRING,
  order_year INT,
  order_month INT,
  order_quarter INT,
  order_day_of_week STRING,
  order_hour INT,
  same_billing_shipping BOOLEAN,
  -- Product attributes (denormalized for query performance)
  product_name STRING,
  category_name STRING,
  brand STRING,
  -- Line item measures
  quantity INT,
  quantity_tier STRING,
  unit_price DECIMAL(14,2),
  unit_cost DECIMAL(15,2),
  line_discount_amount DECIMAL(15,2),
  line_total_amount DECIMAL(28,2),
  line_profit DECIMAL(30,2),
  line_margin DECIMAL(35,2),
  -- Order level measures (repeated for each line)
  order_subtotal DECIMAL(38,2),
  order_tax_amount DECIMAL(38,2),
  order_shipping_cost INT,
  order_discount_amount DECIMAL(11,1),
  order_total_amount DECIMAL(38,2),
  order_net_amount DECIMAL(38,2),
  order_discount_percentage DECIMAL(19,2),
  order_value_tier STRING,
  -- Calculated measures
  line_revenue_contribution_pct DECIMAL(10,2),
  extended_cost DECIMAL(30,2),
  extended_price DECIMAL(28,2),
  -- Flags
  is_line_discounted BOOLEAN,
  has_order_discount BOOLEAN,
  is_first_order BOOLEAN,
  is_weekend_order BOOLEAN,
  -- Audit columns
  order_created_at TIMESTAMP,
  order_updated_at TIMESTAMP,
  line_created_at TIMESTAMP,
  processing_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Transactional order fact table combining order header and line item details at transaction grain';

In [0]:
-- Populate Date Dimension
-- Generate dates from 2020-01-01 to 2030-12-31
INSERT OVERWRITE  dim_date
WITH date_range AS (
  SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) AS date_value
)
SELECT 
  CAST(date_format(date_value, 'yyyyMMdd') AS INT) AS date_key,
  date_value,
  year(date_value) AS year,
  quarter(date_value) AS quarter,
  month(date_value) AS month,
  date_format(date_value, 'MMMM') AS month_name,
  weekofyear(date_value) AS week,
  dayofmonth(date_value) AS day_of_month,
  dayofweek(date_value) AS day_of_week,
  date_format(date_value, 'EEEE') AS day_of_week_name,
  dayofyear(date_value) AS day_of_year,
  CASE WHEN dayofweek(date_value) IN (1, 7) THEN true ELSE false END AS is_weekend,
  false AS is_holiday, -- Can be enhanced with holiday calendar
  year(date_value) AS fiscal_year, -- Adjust based on fiscal year start
  quarter(date_value) AS fiscal_quarter,
  month(date_value) AS fiscal_month,
  -- Relative date flags (will be updated dynamically)
  CASE WHEN date_value = current_date() THEN true ELSE false END AS is_current_day,
  CASE WHEN weekofyear(date_value) = weekofyear(current_date()) AND year(date_value) = year(current_date()) THEN true ELSE false END AS is_current_week,
  CASE WHEN month(date_value) = month(current_date()) AND year(date_value) = year(current_date()) THEN true ELSE false END AS is_current_month,
  CASE WHEN quarter(date_value) = quarter(current_date()) AND year(date_value) = year(current_date()) THEN true ELSE false END AS is_current_quarter,
  CASE WHEN year(date_value) = year(current_date()) THEN true ELSE false END AS is_current_year
FROM date_range;

SELECT COUNT(*) AS date_records_loaded FROM  dim_date;

In [0]:
-- Populate Customer Dimension (SCD Type 2)
INSERT OVERWRITE  dim_customer
SELECT 
  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  first_name,
  last_name,
  full_name,
  email,
  email_domain,
  phone,
  phone_cleaned,
  birth_date,
  age,
  age_group,
  gender,
  address_line1,
  address_line2,
  city,
  state,
  country,
  postal_code,
  registration_date,
  registration_year,
  registration_month,
  customer_tier,
  source_system,
  -- SCD Type 2 columns
  COALESCE(updated_at, created_at, current_date()) AS effective_date,
  to_date('9999-12-31') AS end_date,
  true AS is_current,
  -- Audit columns
  created_at,
  updated_at
FROM  skofy27.silver.customers_clean;

SELECT COUNT(*) AS customer_records_loaded FROM  dim_customer;

In [0]:
-- Populate Product Dimension (SCD Type 2)
INSERT OVERWRITE  dim_product
SELECT 
  ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
  product_id,
  product_name,
  product_name_clean,
  description,
  sku,
  -- Category attributes
  category_id,
  category_name,
  category_path,
  -- Supplier attributes
  supplier_id,
  supplier_name,
  -- Product attributes
  brand,
  brand_category,
  color,
  size,
  weight,
  weight_category,
  dimensions,
  -- Pricing attributes
  price,
  cost,
  profit_margin,
  price_tier,
  -- Status attributes
  is_active,
  product_lifecycle_stage,
  created_at AS created_date,
  days_since_launch,
  -- SCD Type 2 columns
  COALESCE(updated_at, created_at, current_date()) AS effective_date,
  to_date('9999-12-31') AS end_date,
  true AS is_current,
  -- Audit columns
  processing_timestamp AS updated_at
FROM  skofy27.silver.products_clean;

SELECT COUNT(*) AS product_records_loaded FROM  dim_product;

In [0]:
-- Populate Warehouse Dimension (SCD Type 2)
INSERT OVERWRITE  dim_warehouse
SELECT 
  ROW_NUMBER() OVER (ORDER BY warehouse_id, warehouse_region) AS warehouse_key,
  warehouse_id,
  warehouse_region,
  -- SCD Type 2 columns
  COALESCE(created_at, current_date()) AS effective_date,
  to_date('9999-12-31') AS end_date,
  true AS is_current,
  -- Audit columns
  created_at,
  processing_timestamp AS updated_at
FROM (
  SELECT DISTINCT
    warehouse_id,
    warehouse_region,
    created_at,
    processing_timestamp
  FROM  skofy27.silver.inventory_clean
);

SELECT COUNT(*) AS warehouse_records_loaded FROM  dim_warehouse;

In [0]:
-- Populate Sales Fact Table
INSERT OVERWRITE  fact_sales
SELECT 
  oi.order_item_id AS sales_key,
  -- Foreign keys to dimensions
  CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
  dc.customer_key,
  dp.product_key,
  -- Degenerate dimensions
  oi.order_id,
  oi.order_item_id,
  -- Order header attributes
  o.order_status,
  o.order_status_category,
  o.payment_method,
  o.payment_method_category,
  o.payment_status,
  o.same_billing_shipping,
  -- Measures - Quantity
  oi.quantity,
  oi.quantity_tier,
  -- Measures - Amounts
  oi.unit_price,
  oi.unit_cost,
  oi.discount_amount,
  oi.total_amount,
  oi.item_profit,
  oi.item_margin,
  -- Order level amounts
  o.tax_amount AS order_tax_amount,
  o.shipping_cost AS order_shipping_cost,
  o.discount_amount AS order_discount_amount,
  o.total_amount AS order_total_amount,
  o.net_amount AS order_net_amount,
  -- Flags
  oi.is_discounted,
  o.has_discount AS has_order_discount,
  -- Audit columns
  oi.created_at,
  oi.processing_timestamp
FROM  skofy27.silver.order_items_clean oi
INNER JOIN  skofy27.silver.orders_clean o ON oi.order_id = o.order_id
LEFT JOIN  dim_customer dc 
  ON o.customer_id = dc.customer_id AND dc.is_current = true
LEFT JOIN  dim_product dp 
  ON oi.product_id = dp.product_id AND dp.is_current = true;

SELECT COUNT(*) AS sales_records_loaded FROM  fact_sales;

In [0]:
-- Populate Inventory Fact Table
INSERT OVERWRITE  fact_inventory
SELECT 
 i.inventory_id AS inventory_key,
  -- Foreign keys to dimensions
  CAST(date_format(i.last_updated, 'yyyyMMdd') AS INT) AS snapshot_date_key,
  dp.product_key,
  dw.warehouse_key,
  -- Degenerate dimension
  i.inventory_id,
  -- Measures - Quantities
  i.quantity_on_hand,
  i.quantity_reserved,
  i.quantity_available,
  i.reorder_level,
  -- Measures - Metrics
  i.days_of_supply,
  -- Attributes
  i.stock_status,
  i.turnover_category,
  -- Flags
  i.needs_reorder,
  i.overstocked,
  -- Audit columns
  i.last_updated,
  i.created_at,
  i.processing_timestamp
FROM  skofy27.silver.inventory_clean i
LEFT JOIN  dim_product dp 
  ON i.product_id = dp.product_id AND dp.is_current = true
LEFT JOIN  dim_warehouse dw 
  ON i.warehouse_id = dw.warehouse_id AND dw.is_current = true;

SELECT COUNT(*) AS inventory_records_loaded FROM  fact_inventory;

In [0]:
-- Populate Reviews Fact Table
INSERT OVERWRITE  fact_reviews
SELECT 
  r.review_id AS review_key,
  -- Foreign keys to dimensions
  CAST(date_format(r.review_date, 'yyyyMMdd') AS INT) AS review_date_key,
  dc.customer_key,
  dp.product_key,
  -- Degenerate dimension
  r.review_id,
  -- Measures - Ratings
  r.rating,
  r.rating_category,
  r.sentiment_score,
  r.sentiment_category,
  r.helpful_votes,
  r.helpfulness_tier,
  -- Text attributes
  r.review_text,
  r.review_text_length,
  -- Time attributes
  r.days_since_purchase,
  r.review_recency,
  r.review_year,
  r.review_month,
  -- Flags
  r.verified_purchase,
  -- Audit columns
  r.created_at,
  r.processing_timestamp
FROM  skofy27.silver.reviews_clean r
LEFT JOIN  dim_customer dc 
  ON r.customer_id = dc.customer_id AND dc.is_current = true
LEFT JOIN  dim_product dp 
  ON r.product_id = dp.product_id AND dp.is_current = true;

SELECT COUNT(*) AS review_records_loaded FROM  fact_reviews;

In [0]:
-- Populate Web Events Fact Table
INSERT OVERWRITE  fact_web_events
SELECT 
  w.event_id AS event_key,
  -- Foreign keys to dimensions
  CAST(date_format(w.event_date, 'yyyyMMdd') AS INT) AS event_date_key,
  dc.customer_key,
  dp.product_key,
  -- Degenerate dimensions
  w.event_id,
  w.session_id,
  -- Event attributes
  w.event_type,
  w.event_category,
  w.event_hour,
  w.event_day_of_week,
  -- Page attributes
  w.page_url,
  w.page_category,
  w.product_category,
  -- Technical attributes
  w.ip_address,
  w.user_agent,
  w.device_type,
  w.browser,
  w.operating_system,
  -- Referrer attributes
  w.referrer,
  w.referrer_category,
  -- Session attributes
  w.session_duration_bucket,
  -- Flags
  w.is_mobile,
  -- Audit columns
  w.timestamp AS event_timestamp,
  w.created_at,
  w.processing_timestamp
FROM  skofy27.silver.web_events_clean w
LEFT JOIN  dim_customer dc 
  ON w.customer_id = dc.customer_id AND dc.is_current = true
LEFT JOIN  dim_product dp 
  ON w.product_id = dp.product_id AND dp.is_current = true;

SELECT COUNT(*) AS web_event_records_loaded FROM  fact_web_events;

In [0]:
-- Populate Orders Fact Table
INSERT OVERWRITE  fact_orders
SELECT 
  o.order_id AS order_key,
  -- Foreign keys to dimensions
  CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
  dc.customer_key,
  -- Degenerate dimension
  o.order_id,
  -- Order attributes
  o.order_status,
  o.order_status_category,
  o.payment_method,
  o.payment_method_category,
  o.payment_status,
  o.order_year,
  o.order_month,
  o.order_quarter,
  o.order_day_of_week,
  o.order_hour,
  -- Address attributes
  o.shipping_address,
  o.billing_address,
  o.same_billing_shipping,
  -- Measures - Amounts
  o.total_amount,
  o.tax_amount,
  o.shipping_cost,
  o.discount_amount,
  o.net_amount,
  -- Derived measures
  o.order_value_tier,
  o.discount_percentage,
  -- Flags
  o.has_discount,
  -- Audit columns
  o.created_at,
  o.updated_at,
  o.processing_timestamp
FROM  skofy27.silver.orders_clean o
LEFT JOIN  dim_customer dc 
  ON o.customer_id = dc.customer_id AND dc.is_current = true;

SELECT COUNT(*) AS order_records_loaded FROM  fact_orders;

In [0]:
-- Populate Order Transactions Fact Table
INSERT OVERWRITE  fact_order_transactions
WITH customer_first_order AS (
  SELECT 
    customer_id,
    MIN(order_date) AS first_order_date
  FROM  skofy27.silver.orders_clean
  GROUP BY customer_id
)
SELECT 
  oi.order_item_id AS transaction_key,
  -- Foreign keys to dimensions
  CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
  dc.customer_key,
  dp.product_key,
  -- Degenerate dimensions
  oi.order_id,
  oi.order_item_id,
  -- Order header attributes
  o.order_status,
  o.order_status_category,
  o.payment_method,
  o.payment_method_category,
  o.payment_status,
  o.order_year,
  o.order_month,
  o.order_quarter,
  o.order_day_of_week,
  o.order_hour,
  o.same_billing_shipping,
  -- Product attributes (denormalized)
  oi.product_name,
  oi.category_name,
  oi.brand,
  -- Line item measures
  oi.quantity,
  oi.quantity_tier,
  oi.unit_price,
  oi.unit_cost,
  oi.discount_amount AS line_discount_amount,
  oi.total_amount AS line_total_amount,
  oi.item_profit AS line_profit,
  oi.item_margin AS line_margin,
  -- Order level measures
  o.total_amount - o.tax_amount - o.shipping_cost AS order_subtotal,
  o.tax_amount AS order_tax_amount,
  o.shipping_cost AS order_shipping_cost,
  o.discount_amount AS order_discount_amount,
  o.total_amount AS order_total_amount,
  o.net_amount AS order_net_amount,
  o.discount_percentage AS order_discount_percentage,
  o.order_value_tier,
  -- Calculated measures
  ROUND((oi.total_amount / NULLIF(o.total_amount, 0)) * 100, 2) AS line_revenue_contribution_pct,
  oi.quantity * oi.unit_cost AS extended_cost,
  oi.quantity * oi.unit_price AS extended_price,
  -- Flags
  oi.is_discounted AS is_line_discounted,
  o.has_discount AS has_order_discount,
  CASE WHEN o.order_date = cfo.first_order_date THEN true ELSE false END AS is_first_order,
  dd.is_weekend AS is_weekend_order,
  -- Audit columns
  o.created_at AS order_created_at,
  o.updated_at AS order_updated_at,
  oi.created_at AS line_created_at,
  oi.processing_timestamp
FROM  skofy27.silver.order_items_clean oi
INNER JOIN  skofy27.silver.orders_clean o ON oi.order_id = o.order_id
LEFT JOIN  dim_customer dc 
  ON o.customer_id = dc.customer_id AND dc.is_current = true
LEFT JOIN  dim_product dp 
  ON oi.product_id = dp.product_id AND dp.is_current = true
LEFT JOIN  dim_date dd
  ON CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) = dd.date_key
LEFT JOIN customer_first_order cfo
  ON o.customer_id = cfo.customer_id;

SELECT COUNT(*) AS transaction_records_loaded FROM  fact_order_transactions;

In [0]:
-- Verify all tables in gold_dimensional schema
SELECT 
  table_name,
  table_type,
  CASE 
    WHEN table_name LIKE 'dim_%' THEN 'Dimension'
    WHEN table_name LIKE 'fact_%' THEN 'Fact'
    ELSE 'Other'
  END AS table_category
FROM apjtechup.information_schema.tables
WHERE table_schema = 'gold_dimensional'
ORDER BY table_category, table_name;

In [0]:
-- Get record counts for all dimension and fact tables
SELECT 'dim_customer' AS table_name, COUNT(*) AS record_count FROM  dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM  dim_product
UNION ALL
SELECT 'dim_date', COUNT(*) FROM  dim_date
UNION ALL
SELECT 'dim_warehouse', COUNT(*) FROM  dim_warehouse
UNION ALL
SELECT 'fact_orders', COUNT(*) FROM  fact_orders
UNION ALL
SELECT 'fact_order_transactions', COUNT(*) FROM  fact_order_transactions
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM  fact_sales
UNION ALL
SELECT 'fact_inventory', COUNT(*) FROM  fact_inventory
UNION ALL
SELECT 'fact_reviews', COUNT(*) FROM  fact_reviews
UNION ALL
SELECT 'fact_web_events', COUNT(*) FROM  fact_web_events
ORDER BY table_name;

In [0]:
-- Example analytical query using the star schema
-- Top 10 products by revenue with customer and date dimensions
SELECT 
  dp.product_name,
  dp.category_name,
  dp.brand,
  COUNT(DISTINCT fs.customer_key) AS unique_customers,
  SUM(fs.quantity) AS total_quantity_sold,
  SUM(fs.total_amount) AS total_revenue,
  SUM(fs.item_profit) AS total_profit,
  ROUND(AVG(fs.item_margin), 2) AS avg_margin_pct
FROM  fact_sales fs
INNER JOIN  dim_product dp ON fs.product_key = dp.product_key
INNER JOIN  dim_date dd ON fs.order_date_key = dd.date_key
WHERE dd.is_current_year = true
GROUP BY dp.product_name, dp.category_name, dp.brand
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- Example analytical query - Customer purchase behavior by tier
SELECT 
  dc.customer_tier,
  dc.age_group,
  dc.state,
  COUNT(DISTINCT dc.customer_key) AS customer_count,
  COUNT(DISTINCT fs.order_id) AS total_orders,
  SUM(fs.total_amount) AS total_revenue,
  ROUND(AVG(fs.total_amount), 2) AS avg_order_value,
  ROUND(SUM(fs.total_amount) / COUNT(DISTINCT dc.customer_key), 2) AS revenue_per_customer
FROM  dim_customer dc
INNER JOIN  fact_sales fs ON dc.customer_key = fs.customer_key
INNER JOIN  dim_date dd ON fs.order_date_key = dd.date_key
WHERE dc.is_current = true
  AND dd.is_current_year = true
GROUP BY dc.customer_tier, dc.age_group, dc.state
ORDER BY total_revenue DESC
LIMIT 20;

# Star Schema Implementation Complete! ✓

## What was created:

### Dimensions (4 tables):
* **dim_customer** - Customer demographics with SCD Type 2
* **dim_product** - Product catalog with category, supplier, and pricing
* **dim_date** - Date dimension (2020-2030) for time-based analysis
* **dim_warehouse** - Warehouse locations for inventory analysis

### Facts (6 tables):
* **fact_orders** - Order header information with totals and payment details
* **fact_order_transactions** - Transactional grain combining order header and line items with enriched metrics
* **fact_sales** - Sales transactions from order_items_clean + orders_clean
* **fact_inventory** - Inventory snapshots by product and warehouse
* **fact_reviews** - Product reviews and ratings
* **fact_web_events** - Web clickstream and user behavior

## Key Features:
* **Apache Iceberg format** for all tables (ACID transactions, time travel, schema evolution)
* Surrogate keys (auto-generated) for all dimensions
* SCD Type 2 implementation for tracking historical changes
* Foreign key relationships to date dimension
* **No indexes** - Iceberg handles optimization internally
* Optimized for analytical queries with star schema design
* All source tables from apjtechup.silver examined and incorporated
* **Transactional fact table** with denormalized attributes for performance

## Iceberg Benefits:
* Time travel and versioning capabilities
* ACID transactions for reliable data updates
* Schema evolution without table rewrites
* Hidden partitioning for better performance
* Efficient incremental processing
* No need for manual index management

## Next Steps:
1. **Incremental Loads**: Convert INSERT OVERWRITE to MERGE statements for incremental updates
2. **Data Quality**: Add data quality checks and validation rules
3. **Optimization**: Add partitioning specifications for large fact tables
4. **Automation**: Schedule this notebook as a job for regular ETL runs
5. **BI Tools**: Connect Tableau, Power BI, or Databricks SQL dashboards to the star schema
6. **Documentation**: Document business definitions and data lineage
7. **Time Travel**: Leverage Iceberg's time travel for historical analysis

## Usage:
Run all cells in sequence to create and populate the complete star schema in `apjtechup.gold_dimensional`